# S3 from code — boto3

Sign in with your OpenID Connect provider, get a temporary credential from STS
`AssumeRoleWithWebIdentity`, then work with the bucket using boto3 (with pandas and pyarrow for the dataset).
[`s3-pyspark-demo.ipynb`](s3-pyspark-demo.ipynb) does the same steps in PySpark, with the same data and the same results.

The dataset is written the way [Trino](https://trino.io) and [Apache Spark](https://spark.apache.org),
the main engines that query data in S3, expect a table: Parquet, partitioned in the Hive layout.

The notebook is self-contained: the sign-in, the STS call and the credential cache are plain code
in the *Plumbing* cell, with no helper package.

| Step | What it shows |
| --- | --- |
| 1 | Sign in (or reuse the cached credential) |
| 2 | What the credential is: keys, session token, expiry |
| 3 | Connect: every setting the client needs, spelled out |
| 4 | Round-trip one object |
| 5 | Write a dataset as CSV and as partitioned Parquet |
| 6 | Read it back and query it |
| 7 | What the role refuses |
| 8 | Clean up |

Everything this notebook writes goes under one scratch prefix, `s3-demo/<run id>/`, and step 8
deletes it.

### Before you start

- `pip install -r requirements.txt`.
- The same config as the S3 Explorer: `.secrets/credentials.json` (or the JSON, base64-encoded, in
  `S3_ASSUMEROLE_CONFIG_B64`). See [`s3-browser-demo.ipynb`](s3-browser-demo.ipynb) for its format.
- Direct network access to the OIDC provider, STS and the S3 endpoint. The notebook ignores
  `http_proxy`/`https_proxy`.
- The role in the config (`role_arn`) must allow writes for steps 4–6 and 8.

<details><summary><b>Plumbing</b> — config, sign-in, STS and the credential cache (click to read)</summary>

- **Config**: the JSON from `S3_ASSUMEROLE_CONFIG_B64` (base64) if set, otherwise from `.secrets/`.
  Error messages name the source, never its contents.
- **Sign-in**: the OAuth 2.0 device authorization grant (RFC 8628). The endpoints come from the
  provider's OpenID Connect discovery document; `client_secret` is sent only if the config has one.
- **STS**: `AssumeRoleWithWebIdentity` with the provider's ID token, for the config's `role_arn`.
- **Cache**: the credential is kept in `~/.cache/s3-assumerole/` (readable only by you) until the
  session token's `exp`, in the same format as the S3 Explorer, so all three notebooks share it.
- Every call goes direct (`trust_env = False`), never through `https_proxy`.
</details>

In [ ]:
import base64
import json
import os
import re
import time
import webbrowser
import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path

import requests
from IPython.display import Markdown, display

CONFIG_ENV = "S3_ASSUMEROLE_CONFIG_B64"
REQUIRED = ("client_id", "oidc_issuer", "sts_endpoint", "s3_endpoint", "role_arn", "bucket")
CACHE_DIR = Path(os.environ.get("XDG_CACHE_HOME", Path.home() / ".cache")) / "s3-assumerole"
STS_NS = {"sts": "https://sts.amazonaws.com/doc/2011-06-15/"}

http = requests.Session()
http.trust_env = False                 # direct: ignore http(s)_proxy and no_proxy
TIMEOUT = (10, 30)                     # seconds: connect, read. A host that can't be reached fails in 10 s.


def request(method, url, **kwargs):
    """One HTTP call with a connect timeout, so an unreachable host fails instead of hanging."""
    return http.request(method, url, timeout=TIMEOUT, **kwargs)


def load_config(name):
    """(config, where it came from): the env variable if set, else ./.secrets/<name> or ../.secrets/<name>."""
    if os.environ.get(CONFIG_ENV):
        try:
            cfg, source = json.loads(base64.b64decode(os.environ[CONFIG_ENV], validate=True)), f"${CONFIG_ENV}"
        except ValueError:
            raise ValueError(f"${CONFIG_ENV} is set but is not base64-encoded JSON") from None
    else:
        path = next((d / name for d in (Path.cwd() / ".secrets", Path.cwd().parent / ".secrets")
                     if (d / name).is_file()), None)
        if path is None:
            raise FileNotFoundError(f"{name} is not in ./.secrets or ../.secrets, and ${CONFIG_ENV} is not set")
        cfg, source = json.loads(path.read_text()), str(path)
    missing = [k for k in REQUIRED if not cfg.get(k)]
    if missing:
        raise ValueError(f"{source} is missing {', '.join(missing)}")
    return cfg, source


def tier(cfg):
    """dev, alpha or beta if the STS or S3 URL says so, otherwise prod."""
    urls = f"{cfg['sts_endpoint']} {cfg['s3_endpoint']}"
    return next((t for t in ("dev", "alpha", "beta") if f".{t}." in urls), "prod")


def route(via, cfg):
    """(proxies, verify) for one service: direct with the public CAs, or through its *_via proxy."""
    if not via:
        return {}, True
    return {"http": via, "https": via}, cfg.get("ca_bundle", "/etc/ssl/certs/ca-certificates.crt")


def jwt_claims(token):
    """A JWT's claims, for display and expiry. Does not verify the signature."""
    parts = token.split(".")
    if len(parts) != 3:
        return {}
    return json.loads(base64.urlsafe_b64decode(parts[1] + "=" * (-len(parts[1]) % 4)))


def cache_file(cfg):
    slug = re.sub(r"[^A-Za-z0-9_.-]+", "_", cfg["role_arn"].split("role/", 1)[-1])
    return CACHE_DIR / f"sts-{tier(cfg)}-{slug}.json"


MIN_LEFT = 15 * 60       # seconds: don't start a run on a credential that expires halfway through


def cached_credential(cfg):
    """The cached credential for this role and STS, if it has at least MIN_LEFT to go."""
    try:
        c = json.loads(cache_file(cfg).read_text())
    except (OSError, ValueError):
        return None
    fresh = c.get("expires_at", 0) - MIN_LEFT > time.time()
    return c if fresh and c.get("role_arn") == cfg["role_arn"] and c.get("sts_endpoint") == cfg["sts_endpoint"] else None


def save_credential(cfg, creds, subject):
    expiry = [datetime.fromisoformat(creds["Expiration"].replace("Z", "+00:00")).timestamp()]
    if "exp" in (claims := jwt_claims(creds["SessionToken"])):
        expiry.append(claims["exp"])
    entry = {**creds, "env": tier(cfg), "role_arn": cfg["role_arn"], "sts_endpoint": cfg["sts_endpoint"],
             "subject": subject, "issued_at": time.time(), "expires_at": min(expiry)}
    CACHE_DIR.mkdir(mode=0o700, parents=True, exist_ok=True)
    tmp = cache_file(cfg).with_suffix(".tmp")
    with os.fdopen(os.open(tmp, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600), "w") as f:
        json.dump(entry, f)
    tmp.replace(cache_file(cfg))
    return entry


def device_login(cfg):
    """OAuth device authorization grant: shows a link, waits for approval, returns the ID token."""
    print(f"Signing in with {cfg['oidc_issuer']} …", flush=True)
    oidc = request("GET", cfg["oidc_issuer"].rstrip("/") + "/.well-known/openid-configuration").json()
    client = {"client_id": cfg["client_id"]}
    if cfg.get("client_secret"):                       # optional: public clients have none
        client["client_secret"] = cfg["client_secret"]
    resp = request("POST", oidc["device_authorization_endpoint"], data={**client, "scope": "openid"})
    resp.raise_for_status()
    dev = resp.json()
    url = dev.get("verification_uri_complete") or dev["verification_uri"]
    display(Markdown(f"### 👉 [Click here to sign in]({url})\n\nCode: **`{dev['user_code']}`** — waiting for you…"))
    try:
        webbrowser.open(url)
    except webbrowser.Error:
        pass
    interval, deadline = int(dev.get("interval", 5)), time.time() + int(dev.get("expires_in", 600))
    while time.time() < deadline:
        time.sleep(interval)
        body = request("POST", oidc["token_endpoint"], data={
            "grant_type": "urn:ietf:params:oauth:grant-type:device_code", "device_code": dev["device_code"], **client}).json()
        if "error" not in body:
            return body.get("id_token") or body["access_token"]
        if body["error"] == "slow_down":
            interval += 5
        elif body["error"] != "authorization_pending":
            raise RuntimeError(f"Sign-in failed: {body.get('error_description') or body['error']}")
    raise TimeoutError("The sign-in link expired. Run the cell again.")


def assume_role(cfg, token):
    """STS AssumeRoleWithWebIdentity: the provider's token in, a temporary credential out."""
    form = {"Action": "AssumeRoleWithWebIdentity", "Version": "2011-06-15", "RoleArn": cfg["role_arn"],
            "RoleSessionName": f"s3-demo-{os.getpid()}", "WebIdentityToken": token, "DurationSeconds": "3600"}
    if cfg.get("provider_id"):
        form["ProviderId"] = cfg["provider_id"]
    proxies, verify = route(cfg.get("sts_via"), cfg)
    print(f"Approved. Asking STS ({cfg['sts_endpoint']}) for a credential …", flush=True)
    resp = request("POST", cfg["sts_endpoint"], data=form, proxies=proxies, verify=verify)
    try:
        root = ET.fromstring(resp.text)
    except ET.ParseError:
        raise RuntimeError(f"STS refused: HTTP {resp.status_code} {resp.text[:200]}") from None
    if resp.status_code != 200 or root.find(".//sts:Error", STS_NS) is not None:
        raise RuntimeError(f"STS refused: {root.findtext('.//sts:Code', f'HTTP {resp.status_code}', STS_NS)} "
                           f"{root.findtext('.//sts:Message', '', STS_NS)}")
    c = root.find(".//sts:Credentials", STS_NS)
    return {k: c.findtext(f"sts:{k}", namespaces=STS_NS)
            for k in ("AccessKeyId", "SecretAccessKey", "SessionToken", "Expiration")}


def sign_in(cfg):
    """The cached credential if it's still valid, otherwise a device login and a new one from STS."""
    if creds := cached_credential(cfg):
        return creds, "cache"
    token = device_login(cfg)
    return save_credential(cfg, assume_role(cfg, token), jwt_claims(token).get("sub")), "sts"

## Settings

`CONFIG` is the config file in `.secrets/` (`S3_ASSUMEROLE_CONFIG_B64` replaces it when set).
`RUN_ID` keeps each run's files apart.

In [ ]:
import uuid

CONFIG = "credentials.json"        # 👈 the config file in .secrets/ (the environment variable wins if set)

cfg, source = load_config(CONFIG)
BUCKET = cfg["bucket"]
RUN_ID = uuid.uuid4().hex[:8]
PREFIX = f"s3-demo/{RUN_ID}/"

print(f"Environment: {tier(cfg).upper()}  (from {source})")
print(f"  S3        {cfg['s3_endpoint']}")
print(f"  Role      {cfg['role_arn']}")
print(f"  Scratch   s3://{BUCKET}/{PREFIX}")

## 1. Sign in

Reuses the cached credential (shared with the S3 Explorer and the other engineer notebook) if it
has at least 15 minutes left, so it can't expire halfway through the run. Otherwise approve the
sign-in in the browser; the cell finishes by itself.

In [ ]:
creds, how = sign_in(cfg)
until = time.strftime("%H:%M", time.localtime(creds["expires_at"]))
print(("Reused the cached credential." if how == "cache" else "Signed in; STS issued a new credential (now cached).")
      + f" Valid until {until}. Cache: {cache_file(cfg)}")

## 2. What the credential is

A standard AWS temporary credential: an access key id, a secret access key and a session token.
Any S3 client accepts these three. The session token here is a JWT, so its claims can be read;
STS signed them, and the S3 endpoint checks the signature on every request. Long values (such as
encrypted parts) are shortened below, and the secret access key is never shown.

In [ ]:
from datetime import timezone

claims = jwt_claims(creds["SessionToken"])

print(f"Access key id      {creds['AccessKeyId'][:6]}…")
print(f"Secret access key  (hidden)")
print(f"Session token      {len(creds['SessionToken']):,} characters")
print(f"Expires            {datetime.fromtimestamp(creds['expires_at'], timezone.utc):%Y-%m-%d %H:%M:%S} UTC "
      f"({int((creds['expires_at'] - time.time()) // 60)} min left)")
print("\nSession token claims:")
print(json.dumps({k: (f"<{len(v):,} chars>" if isinstance(v, str) and len(v) > 80 else v)
                  for k, v in claims.items()}, indent=2, default=str))

## 3. Connect

A completely standard boto3 S3 client. Only three things are specific to this setup:

- `endpoint_url`: the S3 endpoint instead of AWS.
- Path-style addressing (`s3://endpoint/bucket/key`), since the endpoint's certificate does not
  cover `<bucket>.<endpoint>`.
- `proxies={}`: connect directly. A TLS-intercepting proxy can drop the `Range` header, which breaks
  the signature on ranged GETs.

In [ ]:
import boto3
from botocore.config import Config

proxies, verify = route(cfg.get("s3_endpoint_via"), cfg)    # ({}, True) unless the config names a proxy

s3 = boto3.client(
    "s3",
    endpoint_url=cfg["s3_endpoint"],
    region_name="us-east-1",
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    verify=verify,
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}, proxies=proxies,
                  connect_timeout=10),
)

resp = s3.list_objects_v2(Bucket=BUCKET, Delimiter="/", MaxKeys=20)
print(f"s3://{BUCKET}/ top level:")
for p in resp.get("CommonPrefixes", []):
    print("  ", p["Prefix"])
for o in resp.get("Contents", []):
    print("  ", o["Key"], o["Size"])

## 4. Round-trip one object

Write a small text object, read it back and compare.

In [ ]:
note = f"Written by s3-boto3-demo, run {RUN_ID}\nhello from boto3\n"
key = f"{PREFIX}hello.txt"
s3.put_object(Bucket=BUCKET, Key=key, Body=note.encode(), ContentType="text/plain")

head = s3.head_object(Bucket=BUCKET, Key=key)
back = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read().decode()
print(f"s3://{BUCKET}/{key}: {head['ContentLength']} bytes, {head['ContentType']}")
print("round trip intact:", back == note)
print(back)

## 5. Write a dataset

A seeded sample of trades (the same rows in both notebooks), written twice:

- as one CSV file, and
- as Parquet, partitioned by `trade_date` in the Hive layout (`trade_date=2026-10-19/…`): the
  layout Trino and Spark read as a partitioned table (pandas, DuckDB and Athena read it too).

In [ ]:
import datetime as dt
import random


def sample_trades(n=5000, seed=7):
    rnd = random.Random(seed)
    tickers = ["AAPL", "MSFT", "IBM", "NVDA", "AMZN", "GOOGL"]
    start = dt.date(2026, 10, 19)
    return [{"trade_id": i + 1,
             "trade_date": (start + dt.timedelta(days=i % 5)).isoformat(),
             "ticker": rnd.choice(tickers),
             "side": rnd.choice(["BUY", "SELL"]),
             "quantity": rnd.randint(1, 50) * 100,
             "price": round(rnd.uniform(50, 500), 2)}
            for i in range(n)]


trades = sample_trades()
print(f"{len(trades):,} trades, e.g. {trades[0]}")

In [ ]:
import io

import pandas as pd

df = pd.DataFrame(trades)

# CSV: one object
s3.put_object(Bucket=BUCKET, Key=f"{PREFIX}trades.csv", Body=df.to_csv(index=False).encode(),
              ContentType="text/csv")

# Parquet: one object per trade_date, in the Hive layout
for day, part in df.groupby("trade_date"):
    buf = io.BytesIO()
    part.drop(columns="trade_date").to_parquet(buf, index=False)
    s3.put_object(Bucket=BUCKET, Key=f"{PREFIX}trades_parquet/trade_date={day}/part-0000.parquet",
                  Body=buf.getvalue())

for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=PREFIX):
    for o in page.get("Contents", []):
        print(f"{o['Size']:>9,}  {o['Key']}")

## 6. Read it back and query it

Read the partitioned Parquet back (the partition column comes back from the paths), check the row
count, and total the notional (`quantity × price`) per ticker. Both notebooks print the same table.

In [ ]:
keys = [o["Key"] for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=f"{PREFIX}trades_parquet/")
        for o in page.get("Contents", []) if o["Key"].endswith(".parquet")]
parts = []
for key in keys:
    part = pd.read_parquet(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()))
    part["trade_date"] = key.split("trade_date=", 1)[1].split("/", 1)[0]
    parts.append(part)
back = pd.concat(parts, ignore_index=True)
print(f"{len(back):,} rows read from {len(keys)} partitions; matches what was written: {len(back) == len(trades)}")

back["notional"] = back.quantity * back.price
summary = (back.groupby("ticker")
               .agg(trades=("trade_id", "count"), quantity=("quantity", "sum"), notional=("notional", "sum"))
               .sort_values("notional", ascending=False))
summary["notional"] = summary.notional.map("{:,.2f}".format)
summary

## 7. What the role refuses

The client has no rules of its own. Everything it can't do is refused by the S3 endpoint, based on
the role. Here: a bucket the role doesn't cover (the first of `extra_buckets` in the config, or a
made-up name).

In [ ]:
from botocore.exceptions import ClientError

OTHER_BUCKET = (cfg.get("extra_buckets") or ["a-bucket-this-role-cannot-reach"])[0]
for label, call in [("list", lambda: s3.list_objects_v2(Bucket=OTHER_BUCKET, MaxKeys=1)),
                    ("write", lambda: s3.put_object(Bucket=OTHER_BUCKET, Key=f"{PREFIX}x.txt", Body=b"x"))]:
    try:
        call()
        print(f"{label:6} s3://{OTHER_BUCKET}/: ⚠️ allowed")
    except ClientError as e:
        err = e.response["Error"]
        print(f"{label:6} s3://{OTHER_BUCKET}/: 🛑 {err.get('Code')} — {err.get('Message', '')}")

## 8. Clean up

Deletes everything under this run's scratch prefix.

In [ ]:
deleted = 0
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=PREFIX):
    for o in page.get("Contents", []):
        s3.delete_object(Bucket=BUCKET, Key=o["Key"])
        deleted += 1
print(f"🗑️  deleted {deleted} objects under s3://{BUCKET}/{PREFIX}")

### Long-running jobs

The credential lasts an hour and is not renewed by itself. For longer jobs, give boto3 a
`botocore.credentials.RefreshableCredentials` whose refresh function calls STS again (with a fresh
token from your provider), or run the job as a workload whose identity token is renewed for you,
such as a Kubernetes service account token with `AWS_WEB_IDENTITY_TOKEN_FILE` and `AWS_ROLE_ARN`.